# Using Transfer Learning Models for FER2013 Classification - Part 1
## Imports:

Mixed precision provided only modest benefits during the CNN stage - once transfer learning models were introduced, the reduction in memory usage became much more important because pretrained architectures consume considerably more GPU resources. The use of:
```python
tf.keras.mixed_precision.set_global_policy(
    "mixed_float16"
)
```
therefore became a key factor in allowing larger batch sizes and reducing memory pressure during transfer learning experiments.

In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
import gc

from src.config import EMOTION_LABELS, EMOTION_LOOKUP
from src.training import evaluate_model
from src.model_config import TL_MODELS

tf.get_logger().setLevel("ERROR")

options = tf.data.Options()

options.experimental_optimization.map_parallelization = True
options.experimental_optimization.parallel_batch = True

tf.config.threading.set_inter_op_parallelism_threads(0)
tf.config.threading.set_intra_op_parallelism_threads(0)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

tf.keras.mixed_precision.set_global_policy('mixed_float16')

I0000 00:00:1785129743.337826     143 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785129743.724355     143 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785129743.724414     143 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785129743.725760     143 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.


## Data:

In [2]:
df = pd.read_csv("data/fer2013.csv")

train_df = df[df["Usage"] == "Training"]
train_df["label"] = train_df["emotion"].map(EMOTION_LOOKUP)
val_df = df[df["Usage"] == "PublicTest"]
val_df["label"] = val_df["emotion"].map(EMOTION_LOOKUP)
test_df = df[df["Usage"] == "PrivateTest"]
test_df["label"] = test_df["emotion"].map(EMOTION_LOOKUP)

train_df.to_csv("data/training_data.csv", index=False)
val_df.to_csv("data/validation_data.csv", index=False)
test_df.to_csv("data/testing_data.csv", index=False)

/tmp/ipykernel_143/1510110141.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["label"] = train_df["emotion"].map(EMOTION_LOOKUP)
/tmp/ipykernel_143/1510110141.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df["label"] = val_df["emotion"].map(EMOTION_LOOKUP)
/tmp/ipykernel_143/1510110141.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://p

## Run All Models `num_runs` Times:

This cell contains the main execution loop for the first transfer learning experiment stage. Unlike the CNN stage, where each configuration represented a different custom architecture, the configurations contained within TL_MODELS represent different pretrained feature extractors and fine-tuning strategies. Typical variations include: VGG16, ResNet50, InceptionV3, Frozen feature extraction, Partial layer unfreezing, and Additional dense classification heads.

Each configuration is evaluated using the same training pipeline, allowing direct comparison between transfer learning strategies while keeping the evaluation methodology unchanged.

In [3]:
for model_name, config in TL_MODELS.items():
    try:
        result = evaluate_model(
            model_fn=config["model_fn"],
            model_name=model_name,
            loss=config["loss"],
            class_weight=config["class_weight"],
            model_kwargs=config["model_kwargs"],
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            batch_size=config["batch_size"],
            emotion_labels=EMOTION_LABELS,
            num_runs=5
        )
    except Exception as e:
        print(f"{model_name} failed:")
        print(e)
        
        gc.collect()
        continue

    if result is None:
        print(f"{model_name} skipped")
        continue
        
    del result

    gc.collect()


Training VGG16_Frozen


I0000 00:00:1785129754.616695     143 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785129754.616809     143 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785129754.616842     143 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785129754.816642     143 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785129754.816809     143 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

VGG16_Frozen: Run 1/5 (batch=256)
RAM: 1.62 GB


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
W0000 00:00:1785129757.684520     223 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1785129757.702710     223 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1785129757.704120     223 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1785129757.705468     223 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1785129757.717135     223 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1785129757.718162     223 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1785129757.718917     223 gpu_timer.cc:114] Skipping

VGG16_Frozen: Run 2/5 (batch=256)
RAM: 3.00 GB


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


... (83 outputs left out)

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

InceptionV3_Unfreeze4_Head: Run 5/5 (batch=64)
RAM: 5.89 GB


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

InceptionV3_Unfreeze4_Head complete (best run = 4, best val acc = 0.4606)


One of the major challenges encountered during transfer learning experimentation was the substantially increased memory consumption of pretrained architectures. While the custom CNN models generally trained without major resource issues, the three transfer learning architectures introduced significantly larger memory requirements.

This ultimately led to the development of the dynamic out-of-memory recovery system implemented within evaluate_model(), including: automatic batch-size reduction, repeated retries after memory failures, session clearing, and garbage collection. Without these safeguards, many transfer learning configurations would terminate prematurely.

This is also the main reasons that The transfer learning pipeline is split between two notebooks.

## Evaluation Of First 'Batch':

Although almost structurally identical to the equivalent CNN comparison stage, this cell performs a much more important role within the transfer learning pipeline - during CNN experimentation, the comparison table primarily served as a reporting mechanism, in the transfer learning workflow, however, the resulting ranking table becomes an active part of the experimental design.

The use of:
```python
TL_model_comparison_part1.csv
```
rather than a generic comparison file reflects that it serves as an intermediate result that directly influences the next stage of experimentation, rather than simply recording the outcome of the current stage.

In [4]:
comparison_rows = []

for model_name in TL_MODELS.keys():

    summary = pd.read_csv(
        f"results/{model_name}_summary.csv",
        index_col=0
    )

    report = pd.read_csv(
        f"reports/{model_name}_classification_report.csv",
        index_col=0
    )

    comparison_rows.append({
        "Model": model_name,
        "Mean Validation Accuracy":summary.loc["val_accuracy", "Mean"],
        "Mean Test Accuracy":summary.loc["test_accuracy", "Mean"],
        "Mean Balanced Accuracy":summary.loc["balanced_accuracy", "Mean"],
        "Mean Epochs":summary.loc["epochs", "Mean"],
        "Mean Training Time (s)":summary.loc["training_time_seconds", "Mean"],
        "Macro Precision":report.loc["macro avg", "precision"],
        "Macro Recall":report.loc["macro avg", "recall"],
        "Macro F1":report.loc["macro avg", "f1-score"],
        "Weighted F1":report.loc["weighted avg", "f1-score"],
        "Validation Accuracy Std": summary.loc["val_accuracy", "Std Dev"]
    })

TL_comparison_df = pd.DataFrame(comparison_rows)

TL_comparison_df["Efficiency Score"] = (
    TL_comparison_df["Mean Balanced Accuracy"] /
    TL_comparison_df["Mean Training Time (s)"]
)

TL_comparison_df = TL_comparison_df.sort_values(
    by=[
        "Mean Balanced Accuracy",
        "Macro F1",
        "Mean Validation Accuracy",
        "Mean Training Time (s)",
        "Validation Accuracy Std"
    ],
    ascending=[False, False, False, True, True]
)

TL_comparison_df.to_csv("results/TL_model_comparison_part1.csv", index=False)
TL_comparison_df

,Model,Mean Validation Accuracy,Mean Test Accuracy,Mean Balanced Accuracy,Mean Epochs,Mean Training Time (s),Macro Precision,Macro Recall,Macro F1,Weighted F1,Validation Accuracy Std,Efficiency Score
1,VGG16_Unfreeze4,0.583282,0.552299,0.492009,14.2,42.063620,0.568910,0.534067,0.542142,0.564250,0.005830,0.011697
2,VGG16_Unfreeze4_Head,0.586236,0.550571,0.464437,15.0,45.935211,0.601720,0.466242,0.464812,0.539627,0.001920,0.010111
8,InceptionV3_Unfreeze4_Head,0.457843,0.441070,0.369446,18.8,150.855297,0.516085,0.380205,0.378976,0.434824,0.003246,0.002449
5,ResNet50_Unfreeze4_Head,0.415715,0.418724,0.349588,99.2,407.416983,0.465934,0.366842,0.369181,0.419142,0.004303,0.000858
7,InceptionV3_Unfreeze4,0.424798,0.415882,0.348240,49.6,344.689389,0.484102,0.352578,0.355792,0.402294,0.001690,0.001010
6,InceptionV3_Frozen,0.425132,0.415603,0.346953,79.0,316.002457,0.451808,0.348778,0.351385,0.397609,0.003893,0.001098
0,VGG16_Frozen,0.415102,0.410031,0.334879,231.6,531.741580,0.344080,0.336532,0.332665,0.392619,0.001852,0.000630
4,ResNet50_Unfreeze4,0.347952,0.350627,0.276303,112.0,284.596229,0.293578,0.279150,0.267557,0.321159,0.004911,0.000971
3,ResNet50_Frozen,0.337587,0.341599,0.265180,214.6,540.942201,0.300606,0.264537,0.253446,0.304235,0.001469,0.000490


## Launch And Run Transfer_part2 In Separate Kernel

This first transfer learning notebook evaluates a wide range of pretrained architectures and generates the ranking table used to identify the strongest-performing models. Once those results have been saved, the second notebook can begin automatically without user intervention.

The second stage focuses on applying additional training strategies to the highest-ranked transfer learning models, including: class weighting, focal loss, and weighted focal loss. Rather than requiring manual execution, this code launches the next notebook automatically, creating a semi-automated experimental pipeline.

Launching the second notebook as a separate process ensures that it starts with a completely new Python interpreter and TensorFlow session. This effectively resets GPU memory allocations, TensorFlow graphs, cached objects, notebook state, and reduces the likelihood of memory-related failures during the second stage.

I chose `subprocess.Popen()` rather than `subprocess.run()` as `Popen()` starts the new process and immediately returns control to the current notebook. Meaning the first notebook finishes normally, the second notebook continues independently, and no waiting is required.

In contrast, `run()` would block execution until the second notebook completed.

In [5]:
print("Launching Transfer_part2.ipynb...")

subprocess.Popen([
    "jupyter",
    "nbconvert",
    "--to",
    "notebook",
    "--execute",
    "--inplace",
    "Transfer_part2.ipynb"
])

print("Transfer_part2 started.")

Launching Transfer_part2.ipynb...
Transfer_part2 started.


[NbConvertApp] Converting notebook Transfer_part2.ipynb to notebook
I0000 00:00:1785143115.418623  234490 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785143115.532390  234490 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785143115.532591  234490 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785143115.533976  234490 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1785143126.220227  234490 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:06:00.0/numa_n